In [1]:
import numpy as np
import random

class AutonomousCitySimulator:
    def __init__(self, grid_size=10, initial_budget=1000):
        self.grid_size = grid_size
        self.initial_budget = initial_budget
        
        # Legend Mapping
        self.building_types = {
            0: '.', # Empty
            1: '=', # House
            2: 'R', # Road
            3: 'F', # Farm
            4: 'S', # Hospital
            5: 'P', # Park
            6: 'I', # Factory
            7: 'W', # Water Plant
            8: 'E', # Power Plant
            9: 'T', # Fire Station
            10: 'C' # City Center
        }
        
        self.costs = {
            '=': 50, 'R': 10, 'F': 100, 'S': 200, 'P': 80,
            'I': 250, 'W': 150, 'E': 200, 'T': 150, 'C': 500
        }

        self.reset()

    def reset(self):
        """Resets the environment grid and global metrics."""
        self.grid = np.full((self.grid_size, self.grid_size), '.', dtype=str)
        self.budget = self.initial_budget
        
        # Metrics tracking
        self.population = 0
        self.wellbeing = 50.0  # 0 to 100 scale
        self.stability = 100.0  # Resilience indicator
        self.power_supply = 0
        self.water_supply = 0
        self.food_supply = 0
        self.step_count = 0
        
        # Place City Center in the middle by default
        mid = self.grid_size // 2
        self.grid[mid, mid] = 'C'
        
        return self._get_observation()

    def build(self, x, y, building_type):
        """Executes a build action at (x, y)."""
        if self.grid[x, y] != '.':
            return False, "Cell already occupied."
        
        cost = self.costs.get(building_type, 0)
        if self.budget < cost:
            return False, "Insufficient budget."

        self.grid[x, y] = building_type
        self.budget -= cost
        return True, f"Successfully built {building_type} at ({x}, {y})"

    def _apply_disruptions(self):
        """Simulates random unexpected events (Earthquakes, Fires, Supply Disruptions)."""
        disruption_occurred = False
        disruption_log = ""

        # 5% chance per step for a random disruption
        if random.random() < 0.05:
            disruption_occurred = True
            event_type = random.choice(["FIRE", "INFRASTRUCTURE_FAILURE"])
            
            if event_type == "FIRE":
                # Find all built non-empty cells
                built_cells = np.argwhere(self.grid != '.')
                if len(built_cells) > 0:
                    rx, ry = random.choice(built_cells)
                    # Fire stations mitigate fire damage
                    has_fire_station = np.any(self.grid == 'T')
                    if not has_fire_station:
                        self.grid[rx, ry] = '.'
                        self.stability -= 15
                        disruption_log = f"Fire destroyed building at ({rx}, {ry})!"
                    else:
                        self.stability -= 5
                        disruption_log = f"Fire at ({rx}, {ry}) quickly contained by Fire Station!"

            elif event_type == "INFRASTRUCTURE_FAILURE":
                loss = random.randint(50, 150)
                self.budget = max(0, self.budget - loss)
                self.stability -= 10
                disruption_log = f"Unexpected infrastructure failure! Repair cost: ${loss}"

        return disruption_log

    def step(self, action):
        """
        Advances the city simulator by 1 step (day/month).
        Action format: (x, y, building_symbol)
        """
        x, y, building_symbol = action
        build_success, msg = self.build(x, y, building_symbol)

        # 1. Recalculate City Stats
        houses = np.sum(self.grid == '=')
        farms = np.sum(self.grid == 'F')
        hospitals = np.sum(self.grid == 'S')
        parks = np.sum(self.grid == 'P')
        factories = np.sum(self.grid == 'I')
        water = np.sum(self.grid == 'W')
        power = np.sum(self.grid == 'E')

        self.population = houses * 10
        self.food_supply = farms * 30
        self.water_supply = water * 40
        self.power_supply = power * 50

        # Tax revenue generated by factories and housing
        income = (factories * 40) + (self.population * 2)
        self.budget += income

        # 2. Evaluate Citizen Wellbeing
        food_deficit = max(0, self.population - self.food_supply)
        water_deficit = max(0, self.population - self.water_supply)
        power_deficit = max(0, self.population - self.power_supply)
        
        basic_needs_met = (food_deficit == 0 and water_deficit == 0 and power_deficit == 0)
        
        # Wellbeing adjustments
        wellbeing_delta = 0
        wellbeing_delta += (parks * 2.0)
        wellbeing_delta += (hospitals * 1.5) if self.population > 0 else 0
        if not basic_needs_met:
            wellbeing_delta -= 10.0
            self.stability -= 2.0
        
        self.wellbeing = np.clip(self.wellbeing + wellbeing_delta, 0, 100)
        self.stability = np.clip(self.stability + 0.5, 0, 100)  # Gradual natural recovery

        # 3. Simulate Random Disruptions
        disruption_msg = self._apply_disruptions()

        # 4. Compute Reward (Multi-Objective Optimization)
        reward = (
            (self.wellbeing * 0.4) +
            (self.stability * 0.3) +
            (self.population * 0.2) +
            (self.budget * 0.001)
        )

        self.step_count += 1
        done = self.step_count >= 100 or self.stability <= 0

        info = {
            "msg": msg,
            "disruption": disruption_msg,
            "income": income,
            "reward": reward
        }

        return self._get_observation(), reward, done, info

    def _get_observation(self):
        return {
            "grid": self.grid.copy(),
            "budget": self.budget,
            "population": self.population,
            "wellbeing": self.wellbeing,
            "stability": self.stability,
            "power_supply": self.power_supply,
            "water_supply": self.water_supply,
            "food_supply": self.food_supply
        }

    def render(self):
        """Console visualization of the city map."""
        print(f"\n--- STEP {self.step_count} | Budget: ${self.budget} | Pop: {self.population} | Wellbeing: {self.wellbeing:.1f}% | Stability: {self.stability:.1f}% ---")
        for row in self.grid:
            print(" ".join(row))
        print("-" * 50)


# --- Quick Usage Example ---
if __name__ == "__main__":
    sim = AutonomousCitySimulator(grid_size=6, initial_budget=1000)
    
    # Example sequential build steps
    actions = [
        (0, 0, '='), # Build House
        (0, 1, 'E'), # Build Power Plant
        (0, 2, 'W'), # Build Water Plant
        (0, 3, 'F'), # Build Farm
        (1, 0, 'S'), # Build Hospital
        (1, 1, 'T'), # Build Fire Station
    ]
    
    for action in actions:
        obs, reward, done, info = sim.step(action)
        sim.render()
        if info["disruption"]:
            print(f"⚠️ Event: {info['disruption']}")


--- STEP 1 | Budget: $970 | Pop: 10 | Wellbeing: 40.0% | Stability: 98.5% ---
= . . . . .
. . . . . .
. . . . . .
. . . C . .
. . . . . .
. . . . . .
--------------------------------------------------

--- STEP 2 | Budget: $790 | Pop: 10 | Wellbeing: 30.0% | Stability: 97.0% ---
= E . . . .
. . . . . .
. . . . . .
. . . C . .
. . . . . .
. . . . . .
--------------------------------------------------

--- STEP 3 | Budget: $660 | Pop: 10 | Wellbeing: 20.0% | Stability: 95.5% ---
= E W . . .
. . . . . .
. . . . . .
. . . C . .
. . . . . .
. . . . . .
--------------------------------------------------

--- STEP 4 | Budget: $580 | Pop: 10 | Wellbeing: 20.0% | Stability: 96.0% ---
= E W F . .
. . . . . .
. . . . . .
. . . C . .
. . . . . .
. . . . . .
--------------------------------------------------

--- STEP 5 | Budget: $400 | Pop: 10 | Wellbeing: 21.5% | Stability: 96.5% ---
= E W F . .
S . . . . .
. . . . . .
. . . C . .
. . . . . .
. . . . . .
-----------------------------------------

In [2]:
import numpy as np
import random

class AutonomousCitySimulator30x30:
    def __init__(self, grid_size=30, initial_budget=5000):
        self.grid_size = grid_size
        self.initial_budget = initial_budget
        
        # Legend Mapping
        self.building_types = {
            0: '.',  # Empty Ground
            1: '=',  # House (خانه)
            2: 'R',  # Road (جاده)
            3: 'F',  # Farm (مزرعه)
            4: 'S',  # Hospital (بیمارستان)
            5: 'P',  # Park (پارک)
            6: 'I',  # Factory (کارخانه)
            7: 'W',  # Water Plant / Water Body (آب)
            8: 'E',  # Power Plant (نیروگاه)
            9: 'T',  # Fire Station (ایستگاه آتش‌نشانی)
            10: 'C'  # City Center (مرکز شهر)
        }
        
        self.costs = {
            '=': 50, 'R': 10, 'F': 100, 'S': 300, 'P': 80,
            'I': 300, 'W': 150, 'E': 250, 'T': 200, 'C': 1000
        }

        self.reset()

    def reset(self):
        """Resets grid, time, and global operational metrics."""
        self.grid = np.full((self.grid_size, self.grid_size), '.', dtype=str)
        self.budget = self.initial_budget
        
        # Time System
        self.ticks = 0
        self.day = 1
        self.month = 1
        self.year = 2026

        # Key City Metrics
        self.population = 0
        self.wellbeing = 70.0    # 0 to 100 scale
        self.stability = 100.0   # Structural resilience
        self.power_supply = 0
        self.water_supply = 0
        self.food_supply = 0
        self.healthcare_capacity = 0
        
        # Initialize City Center in the middle (2x2 hub)
        mid = self.grid_size // 2
        self.grid[mid:mid+2, mid:mid+2] = 'C'
        
        return self._get_observation()

    def _advance_time(self):
        """Advances time tracking system."""
        self.ticks += 1
        self.day += 1
        if self.day > 30:
            self.day = 1
            self.month += 1
            if self.month > 12:
                self.month = 1
                self.year += 1

    def build(self, x, y, building_type):
        """Builds structure at coordinate (x, y) if valid."""
        if not (0 <= x < self.grid_size and 0 <= y < self.grid_size):
            return False, "Coordinates out of bounds."
            
        if self.grid[x, y] != '.':
            return False, "Cell already occupied."
        
        cost = self.costs.get(building_type, 0)
        if self.budget < cost:
            return False, "Insufficient budget."

        self.grid[x, y] = building_type
        self.budget -= cost
        return True, f"Successfully built {building_type} at ({x}, {y})"

    def _trigger_disasters(self):
        """Simulates localized environmental and structural disasters."""
        disaster_events = []

        # 8% chance per day/step for an unexpected disaster
        if random.random() < 0.08:
            disaster_type = random.choice(["EARTHQUAKE", "FIRE", "FLOOD", "PANDEMIC"])

            if disaster_type == "EARTHQUAKE":
                cx, cy = random.randint(2, self.grid_size-3), random.randint(2, self.grid_size-3)
                radius = random.randint(1, 3)
                destroyed = 0
                for r in range(-radius, radius + 1):
                    for c in range(-radius, radius + 1):
                        nx, ny = cx + r, cy + c
                        if 0 <= nx < self.grid_size and 0 <= ny < self.grid_size:
                            if self.grid[nx, ny] not in ['.', 'C'] and random.random() < 0.6:
                                self.grid[nx, ny] = '.'
                                destroyed += 1
                self.stability = max(0, self.stability - 15)
                disaster_events.append(f"💥 EARTHQUAKE hit epicentre ({cx}, {cy})! Destroyed {destroyed} structures.")

            elif disaster_type == "FIRE":
                built_cells = np.argwhere((self.grid != '.') & (self.grid != 'W'))
                if len(built_cells) > 0:
                    fx, fy = random.choice(built_cells)
                    # Check for nearby Fire Stations within 4 cells radius
                    has_fire_protection = False
                    for r in range(-4, 5):
                        for c in range(-4, 5):
                            nx, ny = fx + r, fy + c
                            if 0 <= nx < self.grid_size and 0 <= ny < self.grid_size:
                                if self.grid[nx, ny] == 'T':
                                    has_fire_protection = True
                                    break
                    if has_fire_protection:
                        self.stability = max(0, self.stability - 2)
                        disaster_events.append(f"🔥 FIRE at ({fx}, {fy}) was contained by a Fire Station!")
                    else:
                        self.grid[fx, fy] = '.'
                        self.stability = max(0, self.stability - 8)
                        disaster_events.append(f"🔥 FIRE destroyed structure at ({fx}, {fy}) due to lack of fire coverage!")

            elif disaster_type == "FLOOD":
                water_cells = np.argwhere(self.grid == 'W')
                if len(water_cells) > 0:
                    wx, wy = random.choice(water_cells)
                    flooded = 0
                    for dx, dy in [(-1,0), (1,0), (0,-1), (0,1)]:
                        nx, ny = wx + dx, wy + dy
                        if 0 <= nx < self.grid_size and 0 <= ny < self.grid_size:
                            if self.grid[nx, ny] not in ['.', 'W', 'C']:
                                self.grid[nx, ny] = '.'
                                flooded += 1
                    self.stability = max(0, self.stability - 10)
                    disaster_events.append(f"🌊 FLOOD near water source ({wx}, {wy}) washed away {flooded} buildings!")

            elif disaster_type == "PANDEMIC":
                if self.population > 100:
                    deficit = max(0, self.population - self.healthcare_capacity)
                    if deficit > 0:
                        self.wellbeing = max(0, self.wellbeing - 20)
                        self.stability = max(0, self.stability - 10)
                        disaster_events.append(f"🦠 PANDEMIC outbreak! Overwhelmed hospitals. Wellbeing dropped!")
                    else:
                        self.wellbeing = max(0, self.wellbeing - 5)
                        disaster_events.append(f"🦠 PANDEMIC controlled successfully by Hospital system.")

        return disaster_events

    def step(self, action=None):
        """
        Advances the simulation by 1 day.
        Action format (optional): (x, y, building_symbol)
        """
        self._advance_time()
        msg = "No action executed."

        if action:
            x, y, symbol = action
            success, msg = self.build(x, y, symbol)

        # 1. Resource & Infrastructure Recalculations
        houses = np.sum(self.grid == '=')
        farms = np.sum(self.grid == 'F')
        hospitals = np.sum(self.grid == 'S')
        parks = np.sum(self.grid == 'P')
        factories = np.sum(self.grid == 'I')
        water = np.sum(self.grid == 'W')
        power = np.sum(self.grid == 'E')

        self.population = houses * 15
        self.food_supply = farms * 50
        self.water_supply = water * 60
        self.power_supply = power * 75
        self.healthcare_capacity = hospitals * 80

        # Monthly Financial Cycles
        if self.day == 30:
            income = (factories * 100) + (self.population * 5)
            maintenance = (hospitals + power + water) * 20
            self.budget += (income - maintenance)

        # 2. Citizen Wellbeing & Basic Needs Calculation
        food_deficit = max(0, self.population - self.food_supply)
        water_deficit = max(0, self.population - self.water_supply)
        power_deficit = max(0, self.population - self.power_supply)

        wellbeing_delta = (parks * 0.5) - (food_deficit * 0.1 + water_deficit * 0.1 + power_deficit * 0.1)
        self.wellbeing = np.clip(self.wellbeing + wellbeing_delta, 0, 100)
        self.stability = np.clip(self.stability + 0.2, 0, 100)

        # 3. Process Dynamic Disasters
        disasters = self._trigger_disasters()

        # 4. Multi-Objective Optimization Reward Computation
        reward = (
            (self.wellbeing * 0.35) +
            (self.stability * 0.35) +
            (self.population * 0.20) +
            (self.budget * 0.0005)
        )

        done = self.stability <= 0 or self.ticks >= 365  # Simulates 1 full year per episode

        info = {
            "time": f"Year {self.year}, Month {self.month}, Day {self.day}",
            "msg": msg,
            "disasters": disasters,
            "reward": reward
        }

        return self._get_observation(), reward, done, info

    def _get_observation(self):
        return {
            "grid": self.grid.copy(),
            "time": {"year": self.year, "month": self.month, "day": self.day, "ticks": self.ticks},
            "budget": self.budget,
            "population": self.population,
            "wellbeing": round(self.wellbeing, 2),
            "stability": round(self.stability, 2),
            "power_supply": self.power_supply,
            "water_supply": self.water_supply,
            "food_supply": self.food_supply,
            "healthcare_capacity": self.healthcare_capacity
        }

    def render(self):
        """Console visualization showing status and grid map."""
        print(f"\n==================== DATE: {self.year}-{self.month:02d}-{self.day:02d} | Ticks: {self.ticks} ====================")
        print(f"💰 Budget: ${self.budget} | 👥 Pop: {self.population} | 😊 Wellbeing: {self.wellbeing:.1f}% | 🛡️ Stability: {self.stability:.1f}%")
        print(f"⚡ Power: {self.power_supply} | 💧 Water: {self.water_supply} | 🌾 Food: {self.food_supply} | 🏥 Health: {self.healthcare_capacity}")
        print("-" * 65)
        for row in self.grid:
            print("".join(row))
        print("=" * 65)


# --- Simulation Run Example ---
if __name__ == "__main__":
    sim = AutonomousCitySimulator30x30(grid_size=30, initial_budget=10000)
    
    # Run simulation for 60 ticks/days with random constructions
    for step in range(60):
        # Pick random placement
        rx, ry = random.randint(0, 29), random.randint(0, 29)
        building = random.choice(['=', 'R', 'F', 'S', 'P', 'I', 'W', 'E', 'T'])
        
        obs, reward, done, info = sim.step((rx, ry, building))
        
        # Render visual state when major events happen
        if info["disasters"]:
            sim.render()
            for d in info["disasters"]:
                print(d)
        
        if done:
            print("Simulation Ended!")
            break


==================== DATE: 2026-01-10 | Ticks: 9 ====================
💰 Budget: $8610 | 👥 Pop: 15 | 😊 Wellbeing: 68.0% | 🛡️ Stability: 90.0%
⚡ Power: 0 | 💧 Water: 60 | 🌾 Food: 100 | 🏥 Health: 160
-----------------------------------------------------------------
..............................
..............................
.................S....S.......
...........................R..
..............................
..............................
..............................
..............................
..............................
..............................
......................P.......
..............................
..............................
..............................
..............................
...............CC....F........
...............CC.............
..............................
..............................
..............................
...W..........................
..............................
..............................
................I.......

Phase 1: Environment Simulation & Baseline Modeling

    Goal: Build the city simulation grid (including housing, roads, power, water, hospitals, and disasters) and establish the dynamic rule set.

    Key Tasks:

        Map out the grid coordinates, resource generation rate, and cost metrics.

        Implement stochastic disruption events (earthquakes, fires, floods, pandemics).

        Define state tracking metrics (wellbeing, budget, stability, population).

Phase 2: Sequential Deep Learning & Optimization Training

    Goal: Train an AI agent to allocate resources and respond to dynamic city events over time.

    Key Tasks:

        Set up a Deep Reinforcement Learning framework (e.g., Deep Q-Networks or PPO) or sequential decision-making model.

        Formulate multi-objective loss/reward functions balancing wellbeing, budget, and resilience.

        Train the agent to adapt to random, unexpected disruptions without failing.

Phase 3: Strategy Analysis & Reporting (Outputs Phase)

    Goal: Evaluate system stability and produce the required deliverables.

    Key Tasks:

        Generate resource allocation reports and long-term sustainability analysis.

        Evaluate city resilience under varying disaster intensities.

        Finalize decision optimization recommendations.